# X-VLA WidowX Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avikde/vla-pipeline/blob/xvla-actions/notebooks/xvla_widowx_vis_traj.ipynb)

Run X-VLA inference on a simulated WidowX 250s robot in MuJoCo.
The model predicts end-effector trajectories for a pick-and-place task,
visualized as colored dots in the camera views.

**Requirements:** GPU runtime recommended (T4 is sufficient). The model is ~880M params (~1.7GB in fp16).

In [ ]:
import os

# On Colab: clone repo for MuJoCo assets (XML, STL meshes, textures)
# Locally: just cd to the repo root
if os.path.exists('/content'):
    # Running on Colab
    !git clone --depth 1 -b xvla-actions https://github.com/avikde/vla-pipeline.git /content/vla-pipeline 2>/dev/null || echo "Already cloned"
    os.chdir('/content/vla-pipeline')
else:
    # Running locally — assume notebook is in notebooks/ subfolder
    os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
    # Walk up to repo root if needed
    while not os.path.exists('assets/widowx') and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir('..')

assert os.path.exists('assets/widowx/widowx_vision_scene.xml'), \
    f"Cannot find assets/widowx/ from {os.getcwd()}. Run from the repo root."
print(f"Working directory: {os.getcwd()}")

# Install dependencies
!pip install -q "lerobot[xvla]" mujoco Pillow

In [ ]:
import mujoco
import numpy as np
import torch
from PIL import Image, ImageDraw
from IPython.display import display

print(f"MuJoCo: {mujoco.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load X-VLA Policy

In [ ]:
from lerobot.policies.xvla.modeling_xvla import XVLAPolicy
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
policy = XVLAPolicy.from_pretrained("lerobot/xvla-widowx").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(policy.config.tokenizer_name)

print(f"Device: {device}")
print(f"Action mode: {policy.config.action_mode}")
print(f"Chunk size: {policy.config.chunk_size}")
print(f"N action steps: {policy.config.n_action_steps}")

## 2. Load MuJoCo Scene

In [ ]:
xml_path = 'assets/widowx/widowx_vision_scene.xml'
mj_model = mujoco.MjModel.from_xml_path(xml_path)
mj_data = mujoco.MjData(mj_model)

# Initialize to home pose (avoids finger collision at qpos=0)
home_qpos = mj_model.keyframe('home').qpos
n_robot_joints = 8  # 6 arm + 2 finger
mj_data.qpos[:n_robot_joints] = home_qpos[:n_robot_joints]
mj_data.ctrl[:] = mj_model.keyframe('home').ctrl
mujoco.mj_forward(mj_model, mj_data)

# Settle physics
for _ in range(100):
    mujoco.mj_step(mj_model, mj_data)

print(f"Model loaded: nq={mj_model.nq}, nv={mj_model.nv}, nu={mj_model.nu}")

## 3. Helper Functions

In [ ]:
VLA_WIDTH, VLA_HEIGHT = 256, 256
mj_model.vis.global_.offwidth = max(mj_model.vis.global_.offwidth, VLA_WIDTH)
mj_model.vis.global_.offheight = max(mj_model.vis.global_.offheight, VLA_HEIGHT)
renderer = mujoco.Renderer(mj_model, height=VLA_HEIGHT, width=VLA_WIDTH)

# Trajectory marker colors (green -> red gradient)
NUM_MARKERS = 10
MARKER_COLORS = []
for i in range(NUM_MARKERS):
    fade = i / max(1, NUM_MARKERS - 1)
    MARKER_COLORS.append(np.array([fade, 1.0 - fade, 0.0, 0.6], dtype=np.float32))


def render_camera(camera_name, trajectory=None):
    """Render from a camera, optionally with trajectory spheres."""
    camera_id = mj_model.camera(camera_name).id
    renderer.update_scene(mj_data, camera=camera_id)
    if trajectory:
        for i, target_xyz in enumerate(trajectory):
            mujoco.mjv_initGeom(
                renderer.scene.geoms[renderer.scene.ngeom],
                type=mujoco.mjtGeom.mjGEOM_SPHERE,
                size=[0.005, 0, 0],
                pos=target_xyz.astype(np.float64),
                mat=np.eye(3).flatten(),
                rgba=MARKER_COLORS[i],
            )
            renderer.scene.ngeom += 1
    return renderer.render()


def preprocess_image(rgb_image, device='cpu'):
    img_tensor = torch.from_numpy(rgb_image).permute(2, 0, 1).float() / 255.0
    return img_tensor.unsqueeze(0).to(device)


def get_ee_pose():
    ee_body_id = mj_model.body("wx250s/gripper_link").id
    return mj_data.xpos[ee_body_id].copy(), mj_data.xmat[ee_body_id].reshape(3, 3).copy()


def rotation_matrix_to_euler(rot_mat):
    sy = np.sqrt(rot_mat[0, 0]**2 + rot_mat[1, 0]**2)
    if sy > 1e-6:
        roll = np.arctan2(rot_mat[2, 1], rot_mat[2, 2])
        pitch = np.arctan2(-rot_mat[2, 0], sy)
        yaw = np.arctan2(rot_mat[1, 0], rot_mat[0, 0])
    else:
        roll = np.arctan2(-rot_mat[1, 2], rot_mat[1, 1])
        pitch = np.arctan2(-rot_mat[2, 0], sy)
        yaw = 0.0
    return roll, pitch, yaw


def get_ee_state_8d():
    """8D EE state matching BridgeData: [x, y, z, roll, pitch, yaw, pad, gripper]"""
    ee_pos, ee_rot = get_ee_pose()
    roll, pitch, yaw = rotation_matrix_to_euler(ee_rot)
    gripper_pos = mj_data.qpos[mj_model.joint("left_finger").id]
    return np.array([ee_pos[0], ee_pos[1], ee_pos[2],
                     roll, pitch, yaw, 0.0, gripper_pos], dtype=np.float32)


def get_cube_position(cube_name="red_block"):
    try:
        return mj_data.xpos[mj_model.body(cube_name).id].copy()
    except Exception:
        return None


print("Helpers ready.")

## 4. Run Inference

In [ ]:
task_instruction = "Pick up the red block"
tokenized = tokenizer(
    task_instruction,
    padding='max_length',
    max_length=policy.config.tokenizer_max_length,
    truncation=True,
    return_tensors='pt'
)
language_tokens = tokenized['input_ids'].to(device)
language_attention_mask = tokenized['attention_mask'].to(device)

# Initial state
ee_pos, _ = get_ee_pose()
cube_pos = get_cube_position()
print(f"Task: '{task_instruction}'")
print(f"EE position: {ee_pos}")
print(f"Cube position: {cube_pos}")
print(f"Distance: {np.linalg.norm(ee_pos - cube_pos):.3f}m")

In [ ]:
# Run inference until we get the first action chunk
cached_action_targets = []
step = 0

while len(cached_action_targets) == 0:
    img = render_camera('up')
    img2 = render_camera('side')

    img_tensor = preprocess_image(img, device=device)
    img2_tensor = preprocess_image(img2, device=device)
    ee_state_8d = get_ee_state_8d()

    observation = {
        'observation.images.image': img_tensor,
        'observation.images.image2': img2_tensor,
        'observation.state': torch.from_numpy(ee_state_8d).float().unsqueeze(0).to(device),
        'observation.language.tokens': language_tokens,
        'observation.language.attention_mask': language_attention_mask,
    }

    with torch.inference_mode():
        actions = policy.select_action(observation)

    # Check if a new chunk was generated
    action_queue = policy._queues.get("action", [])
    queue_size = len(action_queue)
    is_new_chunk = queue_size == policy.config.chunk_size - 1

    if is_new_chunk:
        for queued_action in list(action_queue)[:NUM_MARKERS]:
            if isinstance(queued_action, torch.Tensor):
                cached_action_targets.append(queued_action.flatten()[:3].cpu().numpy())
            else:
                cached_action_targets.append(np.array(queued_action).flatten()[:3])

    mujoco.mj_step(mj_model, mj_data)
    step += 1
    print(f"Step {step}, queue: {queue_size}", end='\r')

actions_np = actions.detach().cpu().numpy().flatten()
print(f"\nFirst chunk generated at step {step}")
print(f"Cached {len(cached_action_targets)} trajectory targets")
print(f"Action vector (20D): {np.array2string(actions_np, precision=4, suppress_small=True)}")

## 5. Visualize Results

In [ ]:
traj = cached_action_targets
W, H = VLA_WIDTH, VLA_HEIGHT

snap_up = render_camera('up', trajectory=traj)
snap_side = render_camera('side', trajectory=traj)
snap_debug = render_camera('third_person', trajectory=traj)

combined = Image.new('RGB', (W * 3 + 40, H + 30), color=(255, 255, 255))
combined.paste(Image.fromarray(snap_up), (0, 30))
combined.paste(Image.fromarray(snap_side), (W + 20, 30))
combined.paste(Image.fromarray(snap_debug), (W * 2 + 40, 30))

draw = ImageDraw.Draw(combined)
draw.text((W // 2 - 50, 5), 'image (over the shoulder)', fill=(0, 0, 0))
draw.text((W + 20 + W // 2 - 30, 5), 'image2 (side)', fill=(0, 0, 0))
draw.text((W * 2 + 40 + W // 2 - 50, 5), 'debug visualization', fill=(0, 0, 0))

combined.save('camera_views.png')
display(combined)

# Print trajectory info
ee_pos, _ = get_ee_pose()
cube_pos = get_cube_position()
print(f"\nCurrent EE: {np.array2string(ee_pos, precision=4)}")
print(f"Cube:       {np.array2string(cube_pos, precision=4)}")
print(f"\nPredicted trajectory (first {len(traj)} targets):")
for i, t in enumerate(traj):
    dist = np.linalg.norm(t - cube_pos) if cube_pos is not None else 0
    print(f"  [{i}] {np.array2string(t, precision=4)}  dist to cube: {dist:.4f}m")